In [ ]:
import pandas as pd
import numpy as np


df = pd.read_csv('harry_potter_reviews.csv')
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 491 entries, 0 to 490
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   user_id              491 non-null    int64  
 1   user_sex             491 non-null    object 
 2   user_age             491 non-null    int64  
 3   user_country         491 non-null    object 
 4   rating               491 non-null    float64
 5   comment              491 non-null    object 
 6   favourite_character  491 non-null    object 
 7   date                 491 non-null    object 
dtypes: float64(1), int64(2), object(5)
memory usage: 30.8+ KB


In [7]:
df.drop(columns=['user_sex', 'user_age','user_country','rating','date','favourite_character','user_id'], inplace=True)
df.head()

,comment
0,"""The transitions between scenes were awkward, ..."
1,"""Severus Snape's role adds an intriguing layer."""
2,"""The pacing was a bit slow, but the characters..."
3,"""Hagrid's love for magical creatures is heartw..."
4,"""Neville Longbottom's courage is awe-inspiring."""


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 491 entries, 0 to 490
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   comment  491 non-null    object
dtypes: object(1)
memory usage: 4.0+ KB


In [12]:
import string
def basic_text_cleaning(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Convert text to lowercase
    text = text.lower()
    
    # 2. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 3. Remove numbers
    text = ''.join([char for char in text if not char.isdigit()])
    
    # 4. Remove extra whitespaces
    text = ' '.join(text.split())
    
    return text

In [16]:
df['clean_text_basic'] = df['comment'].apply(basic_text_cleaning)

df.head()

,comment,clean_text_basic
0,"""The transitions between scenes were awkward, ...",the transitions between scenes were awkward an...
1,"""Severus Snape's role adds an intriguing layer.""",severus snapes role adds an intriguing layer
2,"""The pacing was a bit slow, but the characters...",the pacing was a bit slow but the characters w...
3,"""Hagrid's love for magical creatures is heartw...",hagrids love for magical creatures is heartwar...
4,"""Neville Longbottom's courage is awe-inspiring.""",neville longbottoms courage is aweinspiring


In [17]:
import re

def advanced_text_cleaning(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Remove URLs
    text = re.sub(r'http[s]?://\S+|www\.\S+', '', text)
    
    # 2. Remove email addresses
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    
    # 3. Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # 4. Remove special characters & emojis (keep only letters, numbers, and spaces for now)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    # Basic cleaning (lowercase + remove extra spaces)
    text = text.lower()
    text = ' '.join(text.split())
    
    return text


df['clean_text_advanced'] = df['comment'].apply(advanced_text_cleaning)

df.head()

,comment,clean_text_basic,clean_text_advanced
0,"""The transitions between scenes were awkward, ...",the transitions between scenes were awkward an...,the transitions between scenes were awkward an...
1,"""Severus Snape's role adds an intriguing layer.""",severus snapes role adds an intriguing layer,severus snapes role adds an intriguing layer
2,"""The pacing was a bit slow, but the characters...",the pacing was a bit slow but the characters w...,the pacing was a bit slow but the characters w...
3,"""Hagrid's love for magical creatures is heartw...",hagrids love for magical creatures is heartwar...,hagrids love for magical creatures is heartwar...
4,"""Neville Longbottom's courage is awe-inspiring.""",neville longbottoms courage is aweinspiring,neville longbottoms courage is aweinspiring


In [22]:
import nltk
from nltk.corpus import stopwords

# nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    if not isinstance(text, str):
        return ""
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)

df['text_no_stopwords'] = df['clean_text_advanced'].apply(remove_stopwords)

df.head(10)

,comment,clean_text_basic,clean_text_advanced,text_no_stopwords
0,"""The transitions between scenes were awkward, ...",the transitions between scenes were awkward an...,the transitions between scenes were awkward an...,transitions scenes awkward soundtrack forgettable
1,"""Severus Snape's role adds an intriguing layer.""",severus snapes role adds an intriguing layer,severus snapes role adds an intriguing layer,severus snapes role adds intriguing layer
2,"""The pacing was a bit slow, but the characters...",the pacing was a bit slow but the characters w...,the pacing was a bit slow but the characters w...,pacing bit slow characters charming
3,"""Hagrid's love for magical creatures is heartw...",hagrids love for magical creatures is heartwar...,hagrids love for magical creatures is heartwar...,hagrids love magical creatures heartwarming
4,"""Neville Longbottom's courage is awe-inspiring.""",neville longbottoms courage is aweinspiring,neville longbottoms courage is aweinspiring,neville longbottoms courage aweinspiring
5,"""Rubeus Hagrid's love for magical creatures is...",rubeus hagrids love for magical creatures is e...,rubeus hagrids love for magical creatures is e...,rubeus hagrids love magical creatures endearing
6,"""Severus Snape's complexity adds depth to the ...",severus snapes complexity adds depth to the story,severus snapes complexity adds depth to the story,severus snapes complexity adds depth story
7,"""Albus Dumbledore's presence feels unnecessary...",albus dumbledores presence feels unnecessary a...,albus dumbledores presence feels unnecessary a...,albus dumbledores presence feels unnecessary d...
8,"""Ron Weasley's humor adds a delightful touch.""",ron weasleys humor adds a delightful touch,ron weasleys humor adds a delightful touch,ron weasleys humor adds delightful touch
9,"""Hermione Granger's determination is inspiring.""",hermione grangers determination is inspiring,hermione grangers determination is inspiring,hermione grangers determination inspiring


In [23]:
def normalize_repeated_chars(text):
    if not isinstance(text, str):
        return text
    # Replace 3 or more repeated characters with single one (except for some common cases)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)   # keeps "sooo" → "soo" (you can change to r'\1' for single)
    # Alternative: make it single character
    # text = re.sub(r'(.)\1{2,}', r'\1', text)
    return text

# 2. Small slang dictionary
slang_dict = {
    'u': 'you',
    'ur': 'your',
    'gr8': 'great',
    'b4': 'before',
    'pls': 'please',
    'thx': 'thanks',
    'lol': 'laughing out loud',
    'omg': 'oh my god',
    # Add more as per your lectures
}

def replace_slang(text):
    if not isinstance(text, str):
        return text
    words = text.split()
    replaced = [slang_dict.get(word, word) for word in words]
    return ' '.join(replaced)

# Apply Task 5 (you can chain them)
df['text_no_stopwords'] = df['text_no_stopwords'].apply(normalize_repeated_chars)
df['text_no_stopwords'] = df['text_no_stopwords'].apply(replace_slang)
df.head(10)

,comment,clean_text_basic,clean_text_advanced,text_no_stopwords
0,"""The transitions between scenes were awkward, ...",the transitions between scenes were awkward an...,the transitions between scenes were awkward an...,transitions scenes awkward soundtrack forgettable
1,"""Severus Snape's role adds an intriguing layer.""",severus snapes role adds an intriguing layer,severus snapes role adds an intriguing layer,severus snapes role adds intriguing layer
2,"""The pacing was a bit slow, but the characters...",the pacing was a bit slow but the characters w...,the pacing was a bit slow but the characters w...,pacing bit slow characters charming
3,"""Hagrid's love for magical creatures is heartw...",hagrids love for magical creatures is heartwar...,hagrids love for magical creatures is heartwar...,hagrids love magical creatures heartwarming
4,"""Neville Longbottom's courage is awe-inspiring.""",neville longbottoms courage is aweinspiring,neville longbottoms courage is aweinspiring,neville longbottoms courage aweinspiring
5,"""Rubeus Hagrid's love for magical creatures is...",rubeus hagrids love for magical creatures is e...,rubeus hagrids love for magical creatures is e...,rubeus hagrids love magical creatures endearing
6,"""Severus Snape's complexity adds depth to the ...",severus snapes complexity adds depth to the story,severus snapes complexity adds depth to the story,severus snapes complexity adds depth story
7,"""Albus Dumbledore's presence feels unnecessary...",albus dumbledores presence feels unnecessary a...,albus dumbledores presence feels unnecessary a...,albus dumbledores presence feels unnecessary d...
8,"""Ron Weasley's humor adds a delightful touch.""",ron weasleys humor adds a delightful touch,ron weasleys humor adds a delightful touch,ron weasleys humor adds delightful touch
9,"""Hermione Granger's determination is inspiring.""",hermione grangers determination is inspiring,hermione grangers determination is inspiring,hermione grangers determination inspiring


In [32]:
# Download required NLTK resources (run once)
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\aryam\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\aryam\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\aryam\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\aryam\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [29]:
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
def perform_tokenization(text):
    if not isinstance(text, str) or text.strip() == "":
        return {"words": [], "sentences": []}
    
    # Word Tokenization
    word_tokens = word_tokenize(text)
    
    # Sentence Tokenization
    sentence_tokens = sent_tokenize(text)
    #
    return {
        "words": word_tokens,
        "sentences": sentence_tokens
    }

In [33]:
samples = df['clean_text_advanced'].head(3)  # Change to .sample(3) if you want random

for i, text in enumerate(samples, 1):
    tokens = perform_tokenization(text)
    print(f"Sample {i}:")
    print("Original:", text[:150] + "..." if len(text) > 150 else text)
    print("Word Tokens:", tokens["words"])
    print("Sentence Tokens:", tokens["sentences"])
    print("-" * 80)
    

Sample 1:
Original: the transitions between scenes were awkward and the soundtrack was forgettable
Word Tokens: ['the', 'transitions', 'between', 'scenes', 'were', 'awkward', 'and', 'the', 'soundtrack', 'was', 'forgettable']
Sentence Tokens: ['the transitions between scenes were awkward and the soundtrack was forgettable']
--------------------------------------------------------------------------------
Sample 2:
Original: severus snapes role adds an intriguing layer
Word Tokens: ['severus', 'snapes', 'role', 'adds', 'an', 'intriguing', 'layer']
Sentence Tokens: ['severus snapes role adds an intriguing layer']
--------------------------------------------------------------------------------
Sample 3:
Original: the pacing was a bit slow but the characters were charming
Word Tokens: ['the', 'pacing', 'was', 'a', 'bit', 'slow', 'but', 'the', 'characters', 'were', 'charming']
Sentence Tokens: ['the pacing was a bit slow but the characters were charming']
-------------------------------------

In [34]:
porter = PorterStemmer()

def apply_stemming(tokens):
    """Apply Porter Stemmer on list of words"""
    stemmed = [porter.stem(word) for word in tokens]
    return stemmed

print("\n=== TASK 7: Stemming (Porter Stemmer) ===\n")

# Take 3 samples for comparison
for i, text in enumerate(df['clean_text_advanced'].head(3), 1):
    word_tokens = word_tokenize(text)
    stemmed_words = apply_stemming(word_tokens)
    
    print(f"Sample {i}:")
    print("Original Words :", word_tokens[:20])   # showing first 20 for readability
    print("Stemmed Words  :", stemmed_words[:20])
    print("-" * 80)

# Optional: Create a new column with stemmed text
df['stemmed_text'] = df['clean_text_advanced'].apply(
    lambda x: ' '.join(apply_stemming(word_tokenize(x))) if isinstance(x, str) else ""
)


=== TASK 7: Stemming (Porter Stemmer) ===

Sample 1:
Original Words : ['the', 'transitions', 'between', 'scenes', 'were', 'awkward', 'and', 'the', 'soundtrack', 'was', 'forgettable']
Stemmed Words  : ['the', 'transit', 'between', 'scene', 'were', 'awkward', 'and', 'the', 'soundtrack', 'wa', 'forgett']
--------------------------------------------------------------------------------
Sample 2:
Original Words : ['severus', 'snapes', 'role', 'adds', 'an', 'intriguing', 'layer']
Stemmed Words  : ['severu', 'snape', 'role', 'add', 'an', 'intrigu', 'layer']
--------------------------------------------------------------------------------
Sample 3:
Original Words : ['the', 'pacing', 'was', 'a', 'bit', 'slow', 'but', 'the', 'characters', 'were', 'charming']
Stemmed Words  : ['the', 'pace', 'wa', 'a', 'bit', 'slow', 'but', 'the', 'charact', 'were', 'charm']
--------------------------------------------------------------------------------


In [35]:
lemmatizer = WordNetLemmatizer()

def apply_lemmatization(tokens):
    """Apply WordNet Lemmatizer"""
    lemmatized = [lemmatizer.lemmatize(word) for word in tokens]
    return lemmatized

print("\n=== TASK 8: Lemmatization vs Stemming ===\n")

for i, text in enumerate(df['clean_text_advanced'].head(3), 1):
    word_tokens = word_tokenize(text)
    stemmed = apply_stemming(word_tokens)
    lemmatized = apply_lemmatization(word_tokens)
    
    print(f"Sample {i}:")
    print("Original   :", word_tokens[:15])
    print("Stemmed    :", stemmed[:15])
    print("Lemmatized :", lemmatized[:15])
    print("-" * 80)

# Create lemmatized column
df['lemmatized_text'] = df['clean_text_advanced'].apply(
    lambda x: ' '.join(apply_lemmatization(word_tokenize(x))) if isinstance(x, str) else ""
)


=== TASK 8: Lemmatization vs Stemming ===

Sample 1:
Original   : ['the', 'transitions', 'between', 'scenes', 'were', 'awkward', 'and', 'the', 'soundtrack', 'was', 'forgettable']
Stemmed    : ['the', 'transit', 'between', 'scene', 'were', 'awkward', 'and', 'the', 'soundtrack', 'wa', 'forgett']
Lemmatized : ['the', 'transition', 'between', 'scene', 'were', 'awkward', 'and', 'the', 'soundtrack', 'wa', 'forgettable']
--------------------------------------------------------------------------------
Sample 2:
Original   : ['severus', 'snapes', 'role', 'adds', 'an', 'intriguing', 'layer']
Stemmed    : ['severu', 'snape', 'role', 'add', 'an', 'intrigu', 'layer']
Lemmatized : ['severus', 'snapes', 'role', 'add', 'an', 'intriguing', 'layer']
--------------------------------------------------------------------------------
Sample 3:
Original   : ['the', 'pacing', 'was', 'a', 'bit', 'slow', 'but', 'the', 'characters', 'were', 'charming']
Stemmed    : ['the', 'pace', 'wa', 'a', 'bit', 'slow', 'but'

In [37]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def nlp_preprocess(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercasing + Noise removal (from Task 3)
    text = text.lower()
    text = re.sub(r'http[s]?://\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = ' '.join(text.split())
    
    # 2. Stopword removal
    words = word_tokenize(text)
    words = [word for word in words if word not in stop_words]
    
    # 3. Lemmatization (preferred over stemming)
    lemmatized = [lemmatizer.lemmatize(word) for word in words]
    
    return ' '.join(lemmatized)

# Apply final pipeline
df['final_clean_text'] = df['comment'].apply(nlp_preprocess)

print("\nFinal Clean Text Column Created Successfully!")
print(df[['comment', 'final_clean_text']].head())


Final Clean Text Column Created Successfully!
                                             comment  \
0  "The transitions between scenes were awkward, ...   
1   "Severus Snape's role adds an intriguing layer."   
2  "The pacing was a bit slow, but the characters...   
3  "Hagrid's love for magical creatures is heartw...   
4   "Neville Longbottom's courage is awe-inspiring."   

                                  final_clean_text  
0  transition scene awkward soundtrack forgettable  
1         severus snapes role add intriguing layer  
2               pacing bit slow character charming  
3       hagrids love magical creature heartwarming  
4         neville longbottoms courage aweinspiring  


# Task 10: Observations & Insights

### 1. Difference between basic and advanced cleaning

- **Basic Cleaning** includes simple operations like converting text to lowercase, removing punctuation, numbers, and extra whitespaces. It is fast and helps in initial noise reduction.
- **Advanced Cleaning** performs deeper noise removal by eliminating URLs, email addresses, HTML tags, emojis, and special characters. 
- **Key Difference**: Basic cleaning handles surface-level noise, while advanced cleaning makes the text much cleaner and more suitable for real-world messy data (social media, web scraped content, etc.).

### 2. Why lemmatization is preferred over stemming

- **Stemming** (Porter Stemmer) is a crude, rule-based method that aggressively cuts suffixes. It is fast but often produces non-real words (e.g., "better" → "bet").
- **Lemmatization** converts words to their actual dictionary base form (lemma) using vocabulary and context. It produces meaningful words (e.g., "better" → "good", "running" → "run").
- **Preference**: Lemmatization is preferred because it maintains better semantic meaning, reduces ambiguity, and gives more accurate results, even though it is slightly slower than stemming.

### 3. Importance of preprocessing in NLP models

Preprocessing is a crucial step in any Natural Language Processing pipeline. It transforms raw, noisy text into clean and structured data. 

**Key Benefits**:
- Removes irrelevant noise and reduces vocabulary size
- Improves model accuracy and performance
- Helps the model focus on meaningful patterns rather than distractions
- Reduces training time and computational resources
- Enhances generalization capability of NLP models

Without proper preprocessing, even the best models can deliver poor results. It forms the foundation for effective feature extraction and model training in tasks like sentiment analysis, text classification, and topic modeling.